# EDA — каталог «Своё вино»

Разведочный анализ каталога и эталонных фото. **Не часть пайплайна** — справочный ноутбук.
Данные: `org_files/strapi_output0709.csv` (каталог) и `filtered/` (обработанные фото).

In [1]:
import sys, re
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# найти корень проекта (где лежит paths.py)
ROOT = Path.cwd()
while not (ROOT / 'paths.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from paths import CATALOG_SRC, FILTERED, CATALOG_CSV
pd.set_option('display.max_colwidth', 60)
print('ROOT =', ROOT)

ROOT = D:\_hack\lct_26


## 1. Каталог: размер и дубли

In [2]:
df = pd.read_csv(CATALOG_SRC, dtype=str, keep_default_na=False)
df.columns = [c.strip() for c in df.columns]
for c in df.columns:
    df[c] = df[c].str.strip()
u = df.drop_duplicates(subset=['Slug']).reset_index(drop=True)
print('строк в CSV        :', len(df))
print('уникальных вин(slug):', u['Slug'].nunique())
print('полных дублей строк :', df.duplicated().sum())
u.head(3)

строк в CSV        : 4147
уникальных вин(slug): 2103
полных дублей строк : 2044


,Название вина,Категория,Цвет,Регион,Сорт винограда,Описание,Винодельня,Slug,Название фото
0,Автохтонное Вино Крыма белое сухое,Белое,Светло-соломенный,Крым,"Алиготе, Кокур Белый, Сары Пандас","В аромате присутствует фруктово-цветочное направление, к...",Валерий Захарьин,avtohtonnoe-vino-kryma-beloe-suhoe,PUSjv7dnMNilH25TO9BcbFN55lPrHsK4eSLja8tm9Ur62n50PyxhlRO1...
1,"Алиготе Баррель, 2024",Белое,Золотистый,Крым,Алиготе,"Вино золотистого цвета с тонкой структурой, хорошей кисл...",Коммуналка,aligote-barrel-2024,DSC09173.webp
2,ЗБ вайн СПУМАНТЕ Брют белое,Белое,соломенный,Крым,Алиготе,"Изысканный букет цитрусовых, с легким нюансом аромата ве...",Золотая Балка,zb-vajn-spumante-bryut-beloe,Спуманте белый брют.webp


## 2. Распределения (по уникальным винам)

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Категория', 'Регион', 'Винодельня']):
    vc = u[col].replace('', '—').value_counts().head(10)
    vc[::-1].plot.barh(ax=ax, color='#8A2B3B')
    ax.set_title(col)
plt.tight_layout(); plt.show()

for col in ['Категория', 'Цвет', 'Регион', 'Винодельня']:
    print(f'{col:12s} — уникальных: {u[col].nunique()}')

Категория    — уникальных: 4
Цвет         — уникальных: 826
Регион       — уникальных: 9
Винодельня   — уникальных: 135


C:\Users\Admin\AppData\Local\Temp\ipykernel_115536\195957520.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3. Сорта винограда (мультизначные через запятую)

In [4]:
grapes = {}
for v in u['Сорт винограда']:
    for g in re.split(r'[;,/]', v):
        g = g.strip()
        if g:
            grapes[g] = grapes.get(g, 0) + 1
gs = pd.Series(grapes).sort_values(ascending=False)
print('уникальных сортов (после split):', len(gs))
gs.head(15)

уникальных сортов (после split): 140


Каберне Совиньон          308
Шардоне                   305
Пино Нуар                 220
Рислинг                   180
Мерло                     177
Совиньон Блан             146
Саперави                  124
Каберне Фран              120
Ркацители                 105
Алиготе                    89
Сира                       61
Красностоп Золотовский     51
Вионье                     43
Рислинг Рейнский           43
Пино Блан                  42
dtype: int64

## 4. Near-duplicates — главный источник ошибок

Серии с одинаковым названием, но разным годом/категорией (одинаковая этикетка).

In [5]:
def base_name(n):
    b = re.sub(r'20\d{2}', '', n.lower())
    return re.sub(r'[^a-zа-яё0-9]+', ' ', b).strip()

u2 = u.copy()
u2['_base'] = u2['Название вина'].map(base_name)
fam = u2.groupby('_base')['Slug'].nunique()
fam = fam[fam > 1].sort_values(ascending=False)
print('серий (одно имя, разный год/категория) с >1 позицией:', len(fam))
print('позиций внутри таких серий:', u2[u2['_base'].isin(fam.index)].shape[0])
fam.head(10)

серий (одно имя, разный год/категория) с >1 позицией:

 111
позиций внутри таких серий: 468


_base
рислинг             37
пино нуар           23
каберне совиньон    21
совиньон блан       20
шардоне             20
мерло               19
каберне фран        17
ркацители           11
саперави            10
розе                 9
Name: Slug, dtype: int64

## 5. Покрытие эталонными фото (filtered/)

In [6]:
cat = pd.read_csv(CATALOG_CSV, dtype=str, keep_default_na=False)
cat['n_images'] = cat['n_images'].astype(int)
print('вин с фото            :', len(cat), f'({100*len(cat)/len(u):.1f}% каталога)')
print('всего фото            :', cat['n_images'].sum())
print('вин с >=2 фото (мультивектор):', (cat['n_images'] >= 2).sum())
cat['n_images'].value_counts().sort_index()

вин с фото            : 1968 (93.6% каталога)
всего фото            : 2044
вин с >=2 фото (мультивектор): 47


n_images
1    1921
2      29
3      10
4       5
5       3
Name: count, dtype: int64

## 6. Разрешение эталонных фото

~⅔ фото — низкого разрешения: потолок для чтения мелкого текста этикетки.

In [7]:
ws = []
for d in FILTERED.iterdir():
    if d.is_dir():
        for p in d.iterdir():
            try:
                ws.append(Image.open(p).size[0])
            except Exception:
                pass
ws = np.array(ws)
print(f'ширина px: медиана={int(np.median(ws))}, <300px={100*(ws<300).mean():.0f}%, <500px={100*(ws<500).mean():.0f}%, max={ws.max()}')
plt.figure(figsize=(8, 3))
plt.hist(np.clip(ws, 0, 1500), bins=50, color='#8A2B3B')
plt.axvline(np.median(ws), color='k', ls='--', label=f'медиана {int(np.median(ws))}px')
plt.title('Распределение ширины эталонных фото'); plt.xlabel('px'); plt.legend(); plt.show()

ширина px: медиана=340, <300px=42%, <500px=64%, max=6337


C:\Users\Admin\AppData\Local\Temp\ipykernel_115536\1522898643.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title('Распределение ширины эталонных фото'); plt.xlabel('px'); plt.legend(); plt.show()


## Выводы

- Уникальных вин: **~2103**; сильный перекос по категориям (Белое/Красное) и регионам (Кубань/Крым).
- **131 near-duplicate серия** (одно имя, разный год/категория) — главный источник ошибок retrieval.
- Эталонные фото есть для **~1968 вин (94%)**; у части — несколько кадров → мультивектор.
- **~65% фото <500px** по ширине — потолок для чтения мелкого текста этикетки
  (⇒ so400m / OCR + веб-скрейп для добора разрешения).